## ML training for paper

### Imports

In [ ]:
from datetime import datetime

import pandas
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from xgboost import XGBRegressor

### Split into train, test, and validation datasets

In [ ]:
# Read in the dataset
total_dataset = pandas.read_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)

In [ ]:
# Initialize empty dataframes for test and validation sets
test_set = pandas.DataFrame()
test_set_indices = []
validation_set = pandas.DataFrame()
validation_set_indices = []

for name, group in total_dataset.groupby("region_code"):
    # Keep track of the last available year for each region
    max_year = group["local_year"].max()

    # Select the test set by selecting the last year
    group_test_set = group[group["local_year"] == max_year].copy()
    test_set_indices.append(group_test_set.index)
    test_set = pandas.concat([test_set, group_test_set], ignore_index=True)

    # Select the validation set by selecting the second last year
    group_val_set = group[group["local_year"] == max_year - 1].copy()
    validation_set_indices.append(group_val_set.index)
    validation_set = pandas.concat(
        [validation_set, group_val_set], ignore_index=True
    )

print(
    "Test set size:",
    round((len(test_set) / len(total_dataset)) * 100, 2),
    "% of total dataset",
)
print(
    "Validation set size:",
    round((len(validation_set) / len(total_dataset)) * 100, 2),
    "% of total dataset",
)

In [ ]:
# Obtain the indicies of the test and validation sets
all_test_set_indices = [
    index for list_indicies in test_set_indices for index in list_indicies
]
all_val_set_indices = [
    index
    for list_indicies in validation_set_indices
    for index in list_indicies
]

In [ ]:
# Drop test and validation sets from the combined dataset
print("Size before:", len(total_dataset))
train_set = total_dataset.drop(index=all_test_set_indices).copy()
train_set = train_set.drop(index=all_val_set_indices)
print("After removing test and val sets:", len(train_set))

In [ ]:
train_set.head()

### Data preparation

In [ ]:
def prepare_data(dataset: pandas.DataFrame):
    """
    Process the dataset into splits to be used in training the model.

    Returns
    -------
    features : pandas.DataFrame
        Features for the model.
    target : pandas.Series
        Column with the target variable.
    groups : pandas.Series
        Column containing the region codes
    """
    features = (
        dataset[
            [
                "local_hour",
                "is_weekend",
                "local_month",
                "year_temp_top1",
                "year_temp_top3",
                "monthly_temp_avg_top1",
                "monthly_temp_avg_rank_top1",
                "year_temp_avg_top1",
                "year_temp_percentile_5",
                "year_temp_percentile_95",
                # "year_electricity_demand_per_capita_mwh",
                # "year_gdp",
            ]
        ]
        .copy(deep=True)
        .reset_index(drop=True)
    )

    categorical_features = [
        "local_hour",
        "is_weekend",
        "local_month",
        "monthly_temp_avg_rank_top1",
    ]

    for cat_feature in categorical_features:
        features[cat_feature] = features[cat_feature].astype("category")

    target = (
        dataset["load_mw_percentage"].copy(deep=True).reset_index(drop=True)
    )
    groups = dataset["region_code"].copy(deep=True).reset_index(drop=True)

    return features, target, groups

In [ ]:
# Generate the train, validation, and test datasets
train_features, train_target, train_groups = prepare_data(train_set)
val_features, val_target, val_groups = prepare_data(validation_set)
test_features, test_target, test_groups = prepare_data(test_set)

### Training

In [ ]:
# Initialize the XGBoost regressor
xgb_model = XGBRegressor(
    random_state=42,
    enable_categorical=True,
    eval_metric=mean_absolute_percentage_error,
)

In [ ]:
# Train the model
xgb_model.fit(
    train_features, train_target, eval_set=[(val_features, val_target)]
)

In [ ]:
# Store the trained model
xgb_model.save_model(
    "./data/xgboost_model" + datetime.now().strftime("%Y_%m_%d_%H%M") + ".bin"
)

In [ ]:
print(xgb_model.feature_names_in_)
print(xgb_model.feature_importances_)

### Prediction on test set

In [ ]:
def calculate_test_error_metric(
    error_metric,
    error_metric_name: str,
    current_predictions,
    current_target,
    current_groups,
    message: str = "",
) -> pandas.DataFrame:
    """
    Calcuate the mean absolute percentage error for the test set
    Saves the results to a parquet and CSV file.

    Returns
    -------
    pandas.DataFrame
        A DataFrame with the region codes, years,
        and mean absolute percentage errors for the test set.
    """
    list_test_metric_values = []
    for name, group in current_groups.groupby("region_code"):
        current_metric = error_metric(
            current_predictions[group.index], current_target.iloc[group.index]
        )

        list_test_metric_values.append([name, current_metric])

    df_validation_metric_values = pandas.DataFrame(
        list_test_metric_values, columns=["region_code", error_metric_name]
    )

    df_validation_metric_values.to_parquet(
        "data/"
        + datetime.now().strftime("%Y_%m_%d_%H%M")
        + "_"
        + error_metric_name
        + "_values"
        + message
        + ".parquet",
        engine="pyarrow",
    )
    df_validation_metric_values.to_csv(
        "data/"
        + datetime.now().strftime("%Y_%m_%d_%H%M")
        + "_"
        + error_metric_name
        + "_values"
        + message
        + ".csv"
    )

    return df_validation_metric_values

In [ ]:
# Predict on test set and calculate MAPE
test_predictions = xgb_model.predict(test_features)
calculate_test_error_metric(
    mean_absolute_percentage_error,
    "MAPE",
    test_predictions,
    test_target,
    test_groups,
    "_test",
)

In [ ]:
# Predict on validation set and calculate MAPE
val_predictions = xgb_model.predict(val_features)
calculate_test_error_metric(
    mean_absolute_percentage_error,
    "MAPE",
    val_predictions,
    val_target,
    val_groups,
    "_val",
)

In [ ]:
# Predict on training set and calculate MAPE
train_predictions = xgb_model.predict(train_features)
calculate_test_error_metric(
    mean_absolute_percentage_error,
    "MAPE",
    train_predictions,
    train_target,
    train_groups,
    "_train",
)

### Cross-validation

In [ ]:
# Read in the dataset
total_dataset = pandas.read_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)

In [ ]:
cv_features, cv_target, cv_groups = prepare_data(total_dataset)

cv_xgb_model = XGBRegressor(
    random_state=42,
    enable_categorical=True,
    eval_metric=mean_absolute_percentage_error,
)

In [ ]:
# Perform cross-validation
cv_results = cross_validate(
    cv_xgb_model,
    cv_features,
    cv_target,
    groups=cv_groups,
    cv=LeaveOneGroupOut(),
    scoring=["neg_mean_absolute_percentage_error"],
    return_train_score=True,
    return_indices=True,
    return_estimator=True,
    n_jobs=1,
)

In [ ]:
cv_results["indices_train"] = cv_results["indices"]["train"]
cv_results["indices_test"] = cv_results["indices"]["test"]

cv_results_filtered = {k: v for k, v in cv_results.items() if k != "indices"}

df_cv_results = pandas.DataFrame(cv_results_filtered)

df_cv_results["test_MAPE"] = -df_cv_results[
    "test_neg_mean_absolute_percentage_error"
]
df_cv_results["train_MAPE"] = -df_cv_results[
    "train_neg_mean_absolute_percentage_error"
]

In [ ]:
df_cv_results.head()

In [ ]:
list_test_group_id = []
for test_indices in cv_results["indices"]["test"]:
    list_test_group_id.append(cv_groups.iloc[test_indices[0]])

df_cv_results["group_id"] = list_test_group_id

In [ ]:
df_cv_output = df_cv_results[
    ["group_id", "train_MAPE", "test_MAPE", "fit_time", "score_time"]
]

In [ ]:
df_cv_output.to_parquet(
    "./data/cv_results"
    + datetime.now().strftime("%Y_%m_%d_%H%M")
    + ".parquet",
    engine="pyarrow",
)
df_cv_output.to_csv(
    "./data/cv_results_" + datetime.now().strftime("%Y_%m_%d_%H%M") + ".csv"
)

### Synthetic data

In [ ]:
# Load data and model
total_dataset = pandas.read_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)
trained_xgb_model = XGBRegressor()
trained_xgb_model.load_model("./data/xgboost_model.bin")

In [ ]:
# Extract features used in model
input_features_columns = trained_xgb_model.feature_names_in_
input_features = total_dataset[input_features_columns]

In [ ]:
predictions = trained_xgb_model.predict(input_features)

In [ ]:
synthetic_dataset = total_dataset.drop(columns=input_features_columns)
synthetic_dataset["predictions"] = predictions
synthetic_dataset = synthetic_dataset.drop(
    columns=[
        "local_year",
        "load_mw",
    ]
)
synthetic_dataset.head()

In [ ]:
synthetic_dataset.to_parquet(
    "./data/synthetic_dataset.parquet", engine="pyarrow"
)

## Visualizations

### Visualize train, test, and validation sets split

In [ ]:
import matplotlib.pyplot as plt

# Get the years from the respective datasets
train_set_years = pandas.DataFrame(
    train_set.groupby("region_code")["local_year"].unique()
)
validation_set_years = pandas.DataFrame(
    validation_set.groupby("region_code")["local_year"].unique()
)
test_set_years = pandas.DataFrame(
    test_set.groupby("region_code")["local_year"].unique()
)

years_per_region = pandas.merge(
    train_set_years,
    validation_set_years,
    on="region_code",
    how="outer",
    suffixes=["_train", "_val"],
)
years_per_region = pandas.merge(
    years_per_region, test_set_years, on="region_code", how="outer"
)
years_per_region = years_per_region.reset_index()

years_per_region.columns = [
    "region_code",
    "train_years",
    "val_years",
    "test_years",
]

In [ ]:
# Calculate total data length across all splits
years_per_region["total_data_length"] = years_per_region.apply(
    lambda row: (
        (
            len(row["train_years"])
            if isinstance(row["train_years"], np.ndarray)
            else 0
        )
        + (
            len(row["val_years"])
            if isinstance(row["val_years"], np.ndarray)
            else 0
        )
        + (
            len(row["test_years"])
            if isinstance(row["test_years"], np.ndarray)
            else 0
        )
    ),
    axis=1,
)
# Sort by total data length
years_per_region = years_per_region.sort_values(
    by="total_data_length", ascending=True
)

# Drop the helper column
years_per_region = years_per_region.drop(columns=["total_data_length"])

# Reset the index
years_per_region = years_per_region.reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots()
list_labels = []
y_min_counter = 0
line_thickness = 0.1


# Ensure the colors will correspond to the legend
train_color = "cornflowerblue"
val_color = "dodgerblue"
test_color = "lightslategray"

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=train_color, label="Train"),
    Patch(facecolor=val_color, label="Validation"),
    Patch(facecolor=test_color, label="Test"),
]


for id, row in years_per_region.iterrows():
    if isinstance(row["train_years"], np.ndarray):
        ax.broken_barh(
            [(int(year), 1) for year in row["train_years"]],
            (y_min_counter, line_thickness),
            color=train_color,
        )
    if isinstance(row["val_years"], np.ndarray):
        ax.broken_barh(
            [(int(year), 1) for year in row["val_years"]],
            (y_min_counter, line_thickness),
            color=val_color,
        )
    if isinstance(row["test_years"], np.ndarray):
        ax.broken_barh(
            [(int(year), 1) for year in row["test_years"]],
            (y_min_counter, line_thickness),
            color=test_color,
        )

    y_min_counter += 0.1

ax.set_xlim(1999, 2026)
ax.set_yticks(range(len(list_labels)), labels=list_labels)
ax.invert_yaxis()
ax.set_title("Training, test, and validation sets splits")
# Add legend with custom handles
ax.legend(handles=legend_elements, loc="upper left")
plt.tight_layout()
plt.savefig("train_val_test_split.jpg", dpi=300)

plt.show()